# 06 - Regularization and model modes

**Goals:** observe dropout, add weight decay and training-only noise, and select weights with early stopping.

Extend the loop from [05](05_training_and_evaluation.ipynb) with regularization choices guided by validation. Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Prepare the data

Recreate the training-fitted tensors from notebook 03. Test rows stay reserved for the final assessment in notebook 08.

In [ ]:
from pathlib import Path
import copy
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Keep these small CPU examples quick.


In [ ]:
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

df = pd.read_csv(data_path)

features = ["feature_1", "feature_2", "feature_3"]
train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]


In [ ]:
scaler = StandardScaler().fit(train[features])  # Fit only on training rows.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)


In [ ]:
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
train_eval_loader = DataLoader(train_data, batch_size=64, shuffle=False)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)


## Observe dropout

During training, dropout randomly zeros entries and scales retained entries by `1 / (1 - p)`. With `p=0.5`, retained ones become twos. Even inside `no_grad()`, training-mode dropout remains random.

In [ ]:
example = torch.ones(2, 3)
dropout = nn.Dropout(p=0.5)
dropout.train()
with torch.no_grad():
    print("training, first call:", dropout(example))
    print("training, second call:", dropout(example))


In evaluation mode dropout returns the inputs unchanged. The two training calls can coincide by chance; evaluation removes this random masking.

In [ ]:
dropout.eval()
print("evaluation:", dropout(example))


## Add regularization to the model

Keep the previous hidden width and add dropout. AdamW applies decoupled weight decay, shrinking parameters during updates. These are separate choices: compare each with its disabled setting rather than assuming the combination improves validation.

In [ ]:
model = nn.Sequential(
    nn.Linear(len(features), 16), nn.ReLU(), nn.Dropout(p=0.25), nn.Linear(16, 2)
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)


Add small Gaussian noise only while training, after standardization. This illustrative augmentation assumes small perturbations preserve the label; that assumption must be checked for real data. Validation uses the original inputs.

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    for x_batch, y_batch in loader:
        augmented = x_batch + 0.05 * torch.randn_like(x_batch)
        optimizer.zero_grad()
        logits = model(augmented)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()


Use the evaluation function from notebook 05 to measure losses on unmodified data with dropout disabled.

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            logits = model(x_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            count += len(y_batch)
    return total_loss / count, correct / count


## Keep the best validation weights

Start with no selected epoch. `patience=5` permits five consecutive epochs without an improvement of more than `1e-4`. A deep copy freezes the selected state; a plain `state_dict()` can still refer to tensors changed by later updates.

In [ ]:
best_loss = float("inf")
best_state = None
best_epoch = 0
patience, stale_epochs = 5, 0


Train for at most 100 epochs, stopping when patience runs out. For a new experiment, recreate the model and optimizer and reset the stopping state above.

In [ ]:
history = []
for epoch in range(1, 101):
    train_one_epoch(model, train_loader)
    train_loss, train_accuracy = evaluate(model, train_eval_loader)
    val_loss, val_accuracy = evaluate(model, val_loader)
    history.append((epoch, train_loss, val_loss, train_accuracy, val_accuracy))
    if val_loss < best_loss - 1e-4:
        best_loss, best_epoch, stale_epochs = val_loss, epoch, 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        stale_epochs += 1
    if stale_epochs >= patience:
        break


The final epoch need not be the selected epoch. Restore the best weights, then set evaluation mode explicitly; the state dictionary does not store the training/evaluation mode.

In [ ]:
model.load_state_dict(best_state)
model.eval()
print(f"Stopped at epoch {epoch}; restored epoch {best_epoch} "
      f"with validation loss {best_loss:.3f}.")


Compare training and validation curves measured without dropout or noise. Early stopping chooses a point on these curves, not necessarily the smallest training loss.

In [ ]:
history_frame = pd.DataFrame(
    history, columns=["epoch", "train_loss", "val_loss", "train_accuracy", "val_accuracy"]
)
history_frame.tail()


In [ ]:
history_frame.plot(x="epoch", y=["train_loss", "val_loss"],
                   ylabel="Cross-entropy loss", title="Learning curves")
plt.show()


## Try it

1. Compare `p=0`, `p=0.25`, and `p=0.6` using validation loss.

2. Try without input noise or weight decay; compare several seeds rather than relying on one run.

3. Remove `eval()` while evaluating dropout. Why can repeated predictions change?

4. Give an example where adding noise would change the correct label and be inappropriate.

**Next:** [Saving and inference](07_saving_and_inference.ipynb) packages model weights and preprocessing for predictions in a new session.